# HPA Dataset — Exploratory Data Analysis

Exploring the Human Protein Atlas dataset for multi-label protein subcellular localization.

- 4-channel fluorescence images (protein, nucleus, microtubule, ER)
- 28 organelle classes, multi-label
- ~31k training images

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
DATA_DIR = '../data'
TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')
TRAIN_DIR = os.path.join(DATA_DIR, 'train')

df = pd.read_csv(TRAIN_CSV)
print(f'Total samples: {len(df)}')
df.head()

## Label Distribution

In [ ]:
from src.data.dataset import HPA_LABELS, NUM_CLASSES

# count per-class frequency
label_counts = Counter()
labels_per_sample = []

for targets in df['Target']:
    labels = str(targets).split()
    labels_per_sample.append(len(labels))
    for lbl in labels:
        label_counts[int(lbl)] += 1

# bar plot
fig, ax = plt.subplots(figsize=(14, 6))
classes = list(range(NUM_CLASSES))
counts = [label_counts.get(c, 0) for c in classes]
bars = ax.bar(classes, counts, color='steelblue')
ax.set_xticks(classes)
ax.set_xticklabels([HPA_LABELS[i] for i in classes], rotation=90)
ax.set_ylabel('Count')
ax.set_title('Class Distribution in HPA Training Set')
plt.tight_layout()
plt.show()

In [ ]:
# labels per sample histogram
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(labels_per_sample, bins=range(1, max(labels_per_sample) + 2),
        edgecolor='black', color='coral', align='left')
ax.set_xlabel('Number of labels per sample')
ax.set_ylabel('Frequency')
ax.set_title('Multi-label cardinality distribution')
plt.tight_layout()
plt.show()

print(f'Mean labels per sample: {np.mean(labels_per_sample):.2f}')
print(f'Max labels per sample: {max(labels_per_sample)}')

## Sample Images

Each image has 4 channels stored as separate PNGs: `{id}_{color}.png`
- **red** = microtubules
- **green** = protein of interest  
- **blue** = nucleus
- **yellow** = endoplasmic reticulum

In [ ]:
def show_sample(image_id, image_dir):
    colors = ['red', 'green', 'blue', 'yellow']
    channel_names = ['Microtubules', 'Protein', 'Nucleus', 'ER']

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    channels = []

    for i, (color, name) in enumerate(zip(colors, channel_names)):
        path = os.path.join(image_dir, f'{image_id}_{color}.png')
        img = np.array(Image.open(path))
        channels.append(img)
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(name)
        axes[i].axis('off')

    # composite RGB (protein=G, nucleus=B, microtubule=R)
    composite = np.stack([channels[0], channels[1], channels[2]], axis=-1)
    composite = (composite / composite.max() * 255).astype(np.uint8)
    axes[4].imshow(composite)
    axes[4].set_title('Composite (RGB)')
    axes[4].axis('off')

    plt.suptitle(f'Sample: {image_id}', y=1.02)
    plt.tight_layout()
    plt.show()

# show first 3 samples
for img_id in df['Id'].values[:3]:
    row = df[df['Id'] == img_id].iloc[0]
    labels = [HPA_LABELS[int(l)] for l in str(row['Target']).split()]
    print(f'Labels: {", ".join(labels)}')
    show_sample(img_id, TRAIN_DIR)

## Per-Channel Statistics

In [ ]:
# compute mean/std for a subset of images
sample_ids = df['Id'].values[:500]
colors = ['red', 'green', 'blue', 'yellow']
channel_stats = {c: {'means': [], 'stds': []} for c in colors}

for img_id in sample_ids:
    for color in colors:
        path = os.path.join(TRAIN_DIR, f'{img_id}_{color}.png')
        if os.path.exists(path):
            img = np.array(Image.open(path), dtype=np.float32) / 255.0
            channel_stats[color]['means'].append(img.mean())
            channel_stats[color]['stds'].append(img.std())

print('Per-channel statistics (first 500 images):')
for color in colors:
    m = np.mean(channel_stats[color]['means'])
    s = np.mean(channel_stats[color]['stds'])
    print(f'  {color:8s} — mean: {m:.4f}, std: {s:.4f}')

## Label Co-occurrence Matrix

In [ ]:
# build co-occurrence matrix
cooccurrence = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)

for targets in df['Target']:
    labels = [int(l) for l in str(targets).split()]
    for i in labels:
        for j in labels:
            cooccurrence[i, j] += 1

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cooccurrence,
    xticklabels=HPA_LABELS,
    yticklabels=HPA_LABELS,
    cmap='YlOrRd',
    ax=ax,
    annot=False,
)
ax.set_title('Label Co-occurrence Matrix')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()